# RS05 · Self-tests, march tests, reliability math and design for test

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/07_reliability_systems/RS05_self_tests_march_tests_reliability_math_and_desi.ipynb)

**Track:** Reliability and systems · **Skill axis (sheet):** Reliability and systems · **Time:** about 50 min

**Skill Hub concepts:** Power-on self-test and march tests; MTBF and exponential reliability; Design for test and manufacturing test

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("RS05")

## Map: proving the hardware is fine, at the factory and at every boot

**Power-on self-test (POST):** CPU registers, RAM (march test), flash CRC, clock check against a second source,
ADC reference, watchdog actually resets. Safety standards (IEC 60730 class B, ISO 13849) expect these.

**March C-** finds stuck-at, transition and most coupling faults in $10n$ operations:
```
 ⇕(w0);  ⇑(r0, w1);  ⇑(r1, w0);  ⇓(r0, w1);  ⇓(r1, w0);  ⇕(r0)        ⇑ ascending addresses, ⇓ descending
```

| Fault | Behavior | Caught by |
|---|---|---|
| Stuck-at 0/1 | cell always reads the same value | any read of the opposite value |
| Transition | cell cannot make one transition (0 -> 1) | read after the failed write |
| Coupling | writing one cell (aggressor) flips another (victim) | reads in both address orders |

**Reliability with a constant failure rate** $\lambda = 1/\text{MTBF}$:
$$R(t) = e^{-t/\text{MTBF}}, \qquad R_{series} = \prod R_i, \qquad R_{parallel} = 1 - \prod (1 - R_i), \qquad
\text{MTBF}_{series} = \Big(\sum 1/\text{MTBF}_i\Big)^{-1}$$

**Design for test:** test points on power rails and buses, a programming/debug header (SWD), boundary scan (JTAG) to
check solder joints without firmware, a factory test firmware or mode, and an end-of-line fixture (bed of nails)
that measures currents, flashes, calibrates and logs results per serial number.

## Task 1: March C- against injected faults
`ram.write(addr, bit)`, `ram.read(addr)`, `ram.n`. Return `True` if the memory passes.

In [ ]:
def march_c_minus(ram):
    raise NotImplementedError  # TODO

lab.check("march", march_c_minus)

In [ ]:
%%writefile rs05_march.c
// C/C++ version of Task 1 (optional, same tests). Declarations: roboprep/c/include/rs05.h
// Grade it with lab.check("march", "rs05_march.c"). For C++, name the file .cpp in both places.
#include "rs05.h"

bool march_c_minus(const RamOps *ram) {
    (void)ram;
    return true; /* TODO: up(w0); up(r0,w1); up(r1,w0); down(r0,w1); down(r1,w0); (r0) */
}

In [ ]:
# lab.check("march", "rs05_march.c")  # uncomment after writing the C/C++ cell above

## Task 2: reliability math

In [ ]:
import math

def reliability(t, mtbf):
    raise NotImplementedError  # TODO

def series(rs):
    raise NotImplementedError  # TODO

def parallel(rs):
    raise NotImplementedError  # TODO

def series_mtbf(mtbfs):
    raise NotImplementedError  # TODO

lab.check("reliability", reliability, series, parallel, series_mtbf)

In [ ]:
%%writefile rs05_reliability.c
// C/C++ version of Task 2 (optional, same tests). Declarations: roboprep/c/include/rs05.h
// Grade it with lab.check("reliability", "rs05_reliability.c"). For C++, name the file .cpp in both places.
#include "rs05.h"

double reliability(double t, double mtbf) { (void)t; (void)mtbf; return 0; /* TODO */ }
double series(const double *rs, size_t n) { (void)rs; (void)n; return 0; /* TODO */ }
double parallel(const double *rs, size_t n) { (void)rs; (void)n; return 0; /* TODO */ }
double series_mtbf(const double *mtbfs, size_t n) { (void)mtbfs; (void)n; return 0; /* TODO */ }

In [ ]:
# lab.check("reliability", "rs05_reliability.c")  # uncomment after writing the C/C++ cell above

## Checkpoint: Math
**M1 (Apply).** A board with MTBF 50,000 h runs one year (8760 h). Probability it survives (4 decimals)?

**M2 (Apply).** Six joint boards in series, each MTBF 100,000 h. System MTBF in hours (whole number)?

**M3 (Apply).** Read and write operations March C- needs for 1024 words?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Analyze).** Why does March C- walk the addresses both up and down?
- a) to run faster
- b) coupling faults depend on whether the aggressor is visited before or after the victim; both orders expose both cases
- c) DRAM requires it

**K2 (Evaluate).** You are designing the Storm joint board for a batch of 50. List the design-for-test features and the
end-of-line test you would add, and what each one catches.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `rs05.h`).
3. In `roboprep/checks/rs05.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()